In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder


#load and merge data accross four tables to usable, format in two dataframes
folder1 = '/Users/alejandrogomez-paz/Desktop/UFC Project/2. data_cleaning/'
folder2 = '/Users/alejandrogomez-paz/Desktop/UFC Project/3. models/ratings_only/optimal_tau_ratings.csv'

df_fighters_fights = pd.read_csv(folder1 + 'fighters_fights.csv')
cols1 = ['fighter_id']
df_fighters_fights = df_fighters_fights.drop(columns = cols1)
df_fighters_fights = df_fighters_fights.sort_values('date', ascending = True)
df_fighters_fights = df_fighters_fights.rename(columns={'fighter_name': 'fighter'})

df_rbr = pd.read_csv(folder1 + 'fights.csv')
df_rbr_agg = df_rbr[df_rbr['round'] == 0]
df_fights = pd.merge(df_rbr_agg, df_fighters_fights, on=['fight_id', 'fighter'])
df_fights = df_fights.fillna(0)
fights_onerow = df_fighters_fights[::2]  



# clean and merge fighter static data with ratings on composite primary key
cols2 = ['slpm',  'sapm', 'td_avg', 'sub_avg', 'wins', 'losses', 'draws', 'str_acc_pct',  'str_def_pct', 'td_acc_pct', 'td_def_pct']
df_fighters = pd.read_csv(folder1 + 'fighters.csv')
df_fighters = df_fighters.drop(columns = cols2)

df_ratings = pd.read_csv(folder2)
df_ratings = df_ratings.rename(columns={'prior_to_date': 'date'})

n = df_fighters['fighter_name'].duplicated().sum()
print(f"{n} rows would be dropped")

df_fighters_u = df_fighters.drop_duplicates('fighter_name', keep='first')
df = pd.merge(df_ratings, df_fighters_u,
                  left_on='fighter', right_on='fighter_name',
                  how='left', validate='m:1')




# time-weighted running average of each stat, EXCLUDING the current fight (no leakage)
df_fights['fight_length_min'] = (((df_fights['round_finished'] - 1) * df_fights['round_time_sec']) + df_fights['stoppage_time_sec']) / 60

df_fights = df_fights.sort_values(['fighter', 'date']).reset_index(drop=True)
g = df_fights.groupby('fighter')

agg = df_fights[['fighter', 'date']].copy()
agg['all_time_min'] = g['fight_length_min'].cumsum().shift(1)
agg.loc[g.cumcount() == 0, 'all_time_min'] = np.nan

cols = ['sig_str_pct', 'td_pct', 'sub_att', 'rev', 'sig_str_landed', 'sig_str_attempted', 'total_str_landed',
       'total_str_attempted', 'td_landed', 'td_attempted', 'sig_str_landed.1', 'sig_str_attempted.1', 'head_landed',
       'head_attempted', 'body_landed', 'body_attempted', 'leg_landed', 'leg_attempted', 'distance_landed',
       'distance_attempted', 'clinch_landed', 'clinch_attempted', 'ground_landed', 'ground_attempted', 'ctrl_secs']

for col in cols:
    wsum = df_fights[col] * df_fights['fight_length_min']
    prior_wsum = wsum.groupby(df_fights['fighter']).cumsum().shift(1)
    prior_wsum[g.cumcount() == 0] = np.nan
    agg[col + '_norm'] = prior_wsum / agg['all_time_min']

agg['wins'] = df_fights.assign(w=(df_fights['fighter'] == df_fights['winner_name'])).groupby('fighter')['w'].cumsum().shift(1)
agg['losses'] = df_fights.assign(l=(df_fights['fighter'] == df_fights['loser_name'])).groupby('fighter')['l'].cumsum().shift(1)

df = df.merge(agg, on=['fighter', 'date'], how='left')



# static fighter biostats normalized by weightclass
cols_to_norm_by_weightclass = ['reach', 'height_inches']
for col in cols_to_norm_by_weightclass:
    grp = df.groupby('weight')[col]
    df[col + '_z'] = (df[col] - grp.transform('mean')) / grp.transform('std')
df = df.drop(columns = cols_to_norm_by_weightclass)



# stance to numeric through One Hot Encoding
counts = df['stance'].value_counts()
rare = counts[counts < df['stance'].isna().sum()].index
df['stance'] = df['stance'].where(~df['stance'].isin(rare), np.nan)
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded = enc.fit_transform(df[['stance']])
df[enc.get_feature_names_out(['stance'])] = encoded


df = df.drop(columns = ['fighter_name', 'fighter_id'])
df['dob'] = pd.to_datetime(df['dob'])
df['date'] = pd.to_datetime(df['date'])
age_years = (df['date'] - df['dob']).dt.days / 365.25
df['age'] = np.floor(pd.to_numeric(age_years, errors='coerce')).astype('Int64')


448 rows would be dropped


In [3]:
df_features = pd.DataFrame()

feature_cols = ['rating', 'rating_deviation', 'volatility', 'sig_str_pct_norm', 'td_pct_norm', 'sub_att_norm', 
                'rev_norm', 'sig_str_landed_norm', 'sig_str_attempted_norm', 'total_str_landed_norm', 'total_str_attempted_norm', 
                'td_landed_norm', 'td_attempted_norm', 'sig_str_landed.1_norm', 'sig_str_attempted.1_norm', 'head_landed_norm',
                  'head_attempted_norm', 'body_landed_norm', 'body_attempted_norm', 'leg_landed_norm', 'leg_attempted_norm', 
                  'distance_landed_norm', 'distance_attempted_norm', 'clinch_landed_norm', 'clinch_attempted_norm', 'ground_landed_norm', 
                  'ground_attempted_norm', 'ctrl_secs_norm', 'wins', 'losses', 'reach_z', 'height_inches_z', 'stance_Orthodox', 
                  'stance_Southpaw', 'stance_Switch', 'stance_nan', 'age']



fights = fights_onerow[['fight_id', 'date', 'fighter', 'opponent_name', 'winner_name']].copy()
fights['date'] = pd.to_datetime(fights['date'])

# drop draws/NCs (~0.4%)
fights = fights[(fights.winner_name == fights.fighter) | (fights.winner_name == fights.opponent_name)]


df_u = df[~df.duplicated(['fighter', 'date'], keep=False)]

fA = fights.merge(df_u, on=['fighter', 'date'], how='left', validate='m:1')
fB = fights.merge(df_u, left_on=['opponent_name', 'date'],
                  right_on=['fighter', 'date'], how='left', validate='m:1')


df_features = fights[['fight_id', 'date']].reset_index(drop=True)
for col in feature_cols:
    df_features[col + '_diff'] = fA[col].values - fB[col].values
df_features['y'] = (fights.winner_name == fights.fighter).astype(int).values
df_features.to_csv('features.csv')